In [1]:
# 데이터 생성
import pandas as pd

dataset = pd.read_csv('../../data/csv/movie_reviews3.csv')

In [2]:
# 독립, 종속
X = dataset.iloc[ : , -1]
# 0.5 => 0으로, 3 => 1로, 5 => 2로
def convert(score):
	return 0 if score == 0.5 else 1 if score == 3 else 2
y = dataset['label'].apply(convert)


In [3]:
# 형태소 분석
from konlpy.tag import Okt
okt = Okt()

def tokenized_korean(text_list):
	return [" ".join(okt.morphs(text, stem=True)) for text in text_list]

morphs_korean = tokenized_korean(X)

In [4]:
# 벡터화
from tensorflow.keras import layers, models
vectorize_layer = layers.TextVectorization(
	max_tokens=1000,
	output_mode='int',
	output_sequence_length = 10
)

vectorize_layer.adapt(morphs_korean)


In [5]:
# 파이프라인
import tensorflow as tf
train_ds = tf.data.Dataset.from_tensor_slices((morphs_korean, y)).batch(8)

In [6]:
# 모델 설계
def build_positional_encoding():
	# 입력층
	inputs = layers.Input((1,), dtype=tf.string)

	#벡터화
	x = vectorize_layer(inputs)
	# 임베딩 : 단어를 
	vocab_size = vectorize_layer.vocabulary_size()
	word_embeding = layers.Embedding(input_dim=vocab_size, output_dim=64)(x)

	# 포지셔널 인코딩
	# 위치 정보 추가 : 0~9 번호를 생성
	positions = tf.range(start=0, limit=10, delta=1)
	# 임베딩 : 위치를 
	pos_embeding = layers.Embedding(input_dim=10, output_dim=64)(positions)

	# 의미 + 위치 
	x = word_embeding + pos_embeding

	# 멀티 헤드 어텐션
	attention_output =layers.MultiHeadAttention(num_heads=2, key_dim=64)(x,x)

	# 잔차 계산 및 정규화
	x = layers.Add()([x, attention_output])
	x = layers.LayerNormalization()(x)

	# ffn
	ffn = layers.Dense(64, activation='relu')(x)
	x = layers.Add()([x, ffn])
	x = layers.LayerNormalization()(x)
	
	# 압축
	x = layers.GlobalAveragePooling1D()(x)
	#  출력층
	ouputs = layers.Dense(3, activation='softmax')(x)
	return models.Model(inputs=inputs, outputs=ouputs)

model = build_positional_encoding()

In [7]:
# 모델 설정
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [8]:
# 모델 학습
model.fit(train_ds, epochs=50, verbose=0)

In [12]:
import numpy as np
# 예측
# 테스트
# test_text = ["액션이 화려해요", "액션이 화려해서 재밌었어요", "액션이 화려하지만 지루해요"]
test_text = ['영화가 너무 재미있어서 하나도 안 지루해요', 
						 '영화가 너무 지루해서 하나도 안 재미있어요',
						 '세련되고 연출이 훌륭한 영화입니다.',
						 '지루하고 재미 없어요',
						 '영화가 너무 자극적이에요',
						 '이야기가 뻔해요',
						 '배우 연기가 별로에요',
						 '절대 안 볼 최악의 영화.']
# 형태소별로 문장을 수정
test_morphs = tokenized_korean(test_text)

# 테스트 할 때 텐서 타입을 변환(문자열 이슈)
test_input = tf.constant(test_morphs)

# 예측 실행
# 0 => 0.5, 1 => 3, 2=>5, 
predictions = model.predict(test_input)
predictions_size = len(predictions)

labels = [0.5, 3, 5]
# numpy의 argmax를 이용
for i in range(predictions_size):
	idx = np.argmax(predictions[i])
	p = predictions[i][idx] * 100
	print(f'{test_text[i]} : {labels[idx]}점({p:.2f}%)')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
영화가 너무 재미있어서 하나도 안 지루해요 : 5점(93.27%)
영화가 너무 지루해서 하나도 안 재미있어요 : 5점(90.95%)
세련되고 연출이 훌륭한 영화입니다. : 5점(99.69%)
지루하고 재미 없어요 : 5점(99.80%)
영화가 너무 자극적이에요 : 5점(99.45%)
이야기가 뻔해요 : 5점(65.54%)
배우 연기가 별로에요 : 3점(81.37%)
절대 안 볼 최악의 영화. : 5점(57.43%)
